# 🕸️ Notebook 03 — Graph Builder
## Financial Representation Learning System

**Purpose:** Build a bipartite customer–counterparty graph from transaction data.

**Key insight:** Individual customer sequences cannot reveal network patterns.
A customer sending money to the same merchant as 200 other customers reveals group behaviour invisible to per-customer analysis.

**Output:** `data/synthetic/graph_data.pkl`

**Runtime:** ~4 minutes (no GPU required)

## Setup

In [ ]:
import os, sys
os.chdir("/content/frl-system")
sys.path.insert(0, ".")

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.graph_builder import BipartiteGraphBuilder, save_graph, load_graph
from src.encoder import load_embeddings
print("✅ Imports ready")

## Step 1 — Load Data

In [ ]:
transactions   = pd.read_csv("data/synthetic/transactions.csv")
customers      = pd.read_csv("data/synthetic/customers.csv")
counterparties = pd.read_csv("data/synthetic/counterparties.csv")
embeddings     = load_embeddings("data/synthetic/customer_embeddings.pkl")

print(f"Transactions   : {len(transactions):,}")
print(f"Customers      : {len(customers):,}")
print(f"Counterparties : {len(counterparties):,}")
print(f"Embeddings     : {len(embeddings):,}")

## Step 2 — Build the Bipartite Graph

In [ ]:
builder = BipartiteGraphBuilder(min_txn=2)
graph_data = builder.build(transactions, customers, counterparties, embeddings)

save_graph(graph_data, "data/synthetic/graph_data.pkl")
print("\n✅ Graph saved to data/synthetic/graph_data.pkl")

## Step 3 — Graph Statistics

In [ ]:
n_nodes = graph_data['x'].shape[0]
n_edges = graph_data['edge_index'].shape[1]
n_customers = graph_data['n_customers']
n_cp = graph_data['n_counterparties']

print("Graph Statistics")
print("=" * 40)
print(f"  Total nodes      : {n_nodes:,}")
print(f"    Customer nodes : {n_customers:,}")
print(f"    Merchant nodes : {n_cp:,}")
print(f"  Total edges      : {n_edges:,} (bidirectional)")
print(f"  Node feature dim : {graph_data['x'].shape[1]}")
print(f"  Edge feature dim : {graph_data['edge_attr'].shape[1]}")
print(f"  Avg degree       : {n_edges/n_nodes:.1f} edges per node")

In [ ]:
# Degree distribution of customer nodes
edge_index = graph_data['edge_index'].numpy()
customer_degree = np.bincount(
    edge_index[0][edge_index[0] < n_customers],
    minlength=n_customers
)

plt.figure(figsize=(10, 4))
plt.hist(customer_degree[customer_degree > 0], bins=50,
         color='#2ECC71', edgecolor='white', linewidth=0.4)
plt.xlabel("Number of Unique Merchants (Degree)")
plt.ylabel("Number of Customers")
plt.title("Customer Node Degree Distribution in Bipartite Graph")
plt.tight_layout()
plt.savefig("docs/results/03_degree_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Customers with 0 edges (fully isolated): {(customer_degree==0).sum():,}")
print(f"These are strong churn/dormancy signals!")

## ✅ Notebook 03 Complete

**What was built:**
- Bipartite graph with customer and counterparty nodes
- Transaction-based edges with amount, channel, and recency features
- Nodes with 64-dim embeddings (from CoLES encoder)
- Degree distribution reveals network connectivity patterns

**Next:** Run `04_gnn_enrichment.ipynb` to train GraphSAGE and enrich the embeddings.